# Hanoi (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Hanoi_cpp.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `Hanoi`: l'algoritmo in C++, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Hanoi_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Hanoi_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno gli stessi nomi dello pseudocodice. Lo stack e i contatori sono variabili globali, e la traccia si attiva con `trace = true`.

## Torri di Hanoi: Hanoi

**Input:** il numero di dischi $n \ge 1$, il piolo di partenza $from$, quello di arrivo $to$ e quello ausiliario $aux$. **Output:** la sequenza delle mosse che porta i dischi $1, \dots, n$ da $from$ a $to$. $M(n) = 2^n - 1$ mosse (ottime), $S(n) = 3 \cdot 2^n - 4 = \Theta(2^n)$ passi, profondità $n$.

Pseudocodice: sezione 14.8 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [1]:
%%writefile hanoi.cpp
#include <cstdarg>
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];      // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
long long calls = 0;    // numero di chiamate = nodi dell'albero delle chiamate
int maxDepth = 0;       // profondita' della ricorsione = altezza massima dello stack
long long mults = 0;    // moltiplicazioni (FastPower)
long long moves = 0;    // mosse (Hanoi)
int ev = 0;             // contatore degli eventi della traccia
int stk[1000];          // lo stack delle chiamate: gli argomenti delle chiamate attive (al massimo 1000)
int sp = 0;             // numero di chiamate attive
long long seen[64];     // seen[x] = numero di chiamate con argomento x
bool trace = false;     // true: stampa la traccia dell'esecuzione

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
    for (int k = 0; k < 64; k++) seen[k] = 0;
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0; sp = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Riga | Esecuzioni\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Passi totali S = %lld\n", Total());
}

string Str(const char* fmt, ...)       // come sprintf, ma restituisce una stringa
{
    char buf[256];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return string(buf);
}

void Push(int x)                       // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stk[sp++] = x;
    if (sp > maxDepth) maxDepth = sp;
    if (x >= 0 && x < 64) seen[x]++;
}

void Pop()                             // la chiamata termina: il suo frame esce dallo stack
{
    sp--;
}

string StackStr()                      // "[4, 3, 2]"
{
    string s = "[";
    for (int k = 0; k < sp; k++)
    {
        if (k > 0) s += ", ";
        s += to_string(stk[k]);
    }
    return s + "]";
}

void Event(const string& text)         // riga numerata della traccia, con lo stack
{
    ev++;
    printf("%2d. %s   stack: %s\n", ev, text.c_str(), StackStr().c_str());
}

void Say(const string& call, const string& action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    printf("%-28s| %s\n", (string(2 * (sp - 1), ' ') + call).c_str(), action.c_str());
}

bool showMoves = false;                            // true: stampa le mosse

void Move(int disk, const char* from, const char* to)   // "move disk d from from to to": conta la mossa e, se showMoves, la stampa
{
    moves++;
    if (showMoves) printf("%lld. sposta il disco %d da %s a %s\n", moves, disk, from, to);
}

// Hanoi(n, from, to, aux)
void Hanoi(int n, const char* from, const char* to, const char* aux)
{
    Push(n); cnt[2]++;
    string call = trace ? Str("Hanoi(%d,%s,%s,%s)", n, from, to, aux) : "";
    if (n == 1)
    {
        cnt[3]++;
        Move(1, from, to);
        if (trace) Say(call, Str("stampa: sposta il disco 1 da %s a %s, ritorna", from, to));
    }
    else
    {
        if (trace) Say(call, Str("chiama Hanoi(%d,%s,%s,%s)", n - 1, from, aux, to));
        cnt[5]++;
        Hanoi(n - 1, from, aux, to);
        cnt[6]++;
        Move(n, from, to);
        if (trace) Say(call, Str("stampa: sposta il disco %d da %s a %s, chiama Hanoi(%d,%s,%s,%s)", n, from, to, n - 1, aux, to, from));
        cnt[7]++;
        Hanoi(n - 1, aux, to, from);
        if (trace) Say(call, "ritorna");
    }
    Pop();
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Esempio delle dispense: Hanoi(3, A, C, B), le mosse
    Reset();
    showMoves = true;
    Hanoi(3, "A", "C", "B");
    showMoves = false;
    printf("%lld mosse, %lld chiamate, profondita' della ricorsione = %d\n", moves, calls, maxDepth);

    // La stessa esecuzione, chiamata per chiamata (rientro = profondita' nello stack)
    printf("\n");
    Reset();
    trace = true;
    printf("%-28s| %s\n", "chiamata", "azione");
    Hanoi(3, "A", "C", "B");
    trace = false;
    PrintSteps({2, 3, 5, 6, 7});
    printf("S(n) = 3 * 2^n - 4 = %d\n", 3 * (1 << 3) - 4);

    // Mosse e passi: M(n) = 2^n - 1 e S(n) = 3 * 2^n - 4
    printf("\n");
    printf("  n |  mosse | 2^n - 1 | chiamate |   S(n) | 3 * 2^n - 4 | profondita'\n");
    for (int k = 1; k <= 10; k++)
    {
        Reset();
        Hanoi(k, "A", "C", "B");
        printf("%3d | %6lld | %7d | %8lld | %6lld | %11d | %d\n", k, moves, (1 << k) - 1, calls, Total(), 3 * (1 << k) - 4, maxDepth);
    }

    // La leggenda: 64 dischi, una mossa al secondo
    printf("\n");
    printf("n = 64: 2^64 - 1 secondi sono piu' di %.1e anni\n", (pow(2.0, 64) - 1) / (365.25 * 24 * 3600));
    return 0;
}
#endif

Writing hanoi.cpp


In [2]:
!g++ -std=c++17 -O2 -o hanoi hanoi.cpp && ./hanoi

1. sposta il disco 1 da A a C
2. sposta il disco 2 da A a B
3. sposta il disco 1 da C a B
4. sposta il disco 3 da A a C
5. sposta il disco 1 da B a A
6. sposta il disco 2 da B a C
7. sposta il disco 1 da A a C
7 mosse, 7 chiamate, profondita' della ricorsione = 3

chiamata                    | azione
Hanoi(3,A,C,B)              | chiama Hanoi(2,A,B,C)
  Hanoi(2,A,B,C)            | chiama Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | stampa: sposta il disco 1 da A a C, ritorna
  Hanoi(2,A,B,C)            | stampa: sposta il disco 2 da A a B, chiama Hanoi(1,C,B,A)
    Hanoi(1,C,B,A)          | stampa: sposta il disco 1 da C a B, ritorna
  Hanoi(2,A,B,C)            | ritorna
Hanoi(3,A,C,B)              | stampa: sposta il disco 3 da A a C, chiama Hanoi(2,B,C,A)
  Hanoi(2,B,C,A)            | chiama Hanoi(1,B,A,C)
    Hanoi(1,B,A,C)          | stampa: sposta il disco 1 da B a A, ritorna
  Hanoi(2,B,C,A)            | stampa: sposta il disco 2 da B a C, chiama Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)

**Prova tu.** Cambia il numero di dischi e conta le mosse.

In [3]:
%%writefile prova_hanoi.cpp
#define SOLO_LIBRERIA   // non eseguire il main di hanoi.cpp
#include "hanoi.cpp"

int main()
{
    int n = 4;        // <-- cambia il numero di dischi
    Reset();
    showMoves = true;
    Hanoi(n, "A", "C", "B");
    showMoves = false;
    printf("%lld mosse = 2^%d - 1 = %d\n", moves, n, (1 << n) - 1);
    return 0;
}

Writing prova_hanoi.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_hanoi prova_hanoi.cpp && ./prova_hanoi

1. sposta il disco 1 da A a B
2. sposta il disco 2 da A a C
3. sposta il disco 1 da B a C
4. sposta il disco 3 da A a B
5. sposta il disco 1 da C a A
6. sposta il disco 2 da C a B
7. sposta il disco 1 da A a B
8. sposta il disco 4 da A a C
9. sposta il disco 1 da B a C
10. sposta il disco 2 da B a A
11. sposta il disco 1 da C a A
12. sposta il disco 3 da B a C
13. sposta il disco 1 da A a B
14. sposta il disco 2 da A a C
15. sposta il disco 1 da B a C
15 mosse = 2^4 - 1 = 15


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).